# 🚀 OMEGA vLLM Server — GPU Edition (Kaggle T4/P100 Optimized)

**Optimized for:** Kaggle GPU (T4 16GB / P100 16GB) — 9hr max session, 30hr/week quota
**Model:** Qwen/Qwen2-VL-7B-Instruct-AWQ (4-bit quantized)
**Features:**
- Rate-limited proxy to prevent abuse & Kaggle throttling
- Idle GPU auto-pause to save GPU hours
- Cloudflare tunnel with health monitoring
- Aggressive memory limits to stay under 16GB ceiling
- Optimized for Kaggle's /kaggle/working persistence

**⚠️ Kaggle Reality Check:**
- Free tier: ~9hr max session, 30 GPU hrs/week
- Sessions auto-shutdown on inactivity (~60 min)
- /kaggle/working persists, /tmp does NOT across sessions
- These settings maximize your *probability* of reaching the limit

**Setup:** `Settings → Accelerator → GPU T4 x2` or `GPU P100 x2` before running!

In [ ]:
# Cell 1: GPU Check & Environment Setup
!nvidia-smi
# Expected: Tesla T4 (15109MiB) or P100 (16280MiB) with ~16GB VRAM
# If CPU only, go to Settings → Accelerator → GPU

# Install compatible transformers for vLLM 0.6.4
!pip install -q "transformers==4.46.3" "tokenizers>=0.19,<0.21" "huggingface_hub>=0.23.0"

from transformers.models.qwen2_vl.image_processing_qwen2_vl import make_batched_images
print("✅ make_batched_images OK")

import transformers, pathlib
fp8_path = pathlib.Path(transformers.__file__).parent / "integrations/finegrained_fp8.py"
print(f"fp8 file exists: {fp8_path.exists()}")   # should print False

# Kaggle-specific: ensure working dir for persistence
import os
os.makedirs("/kaggle/working/vllm_logs", exist_ok=True)
os.makedirs("/kaggle/working/models", exist_ok=True)
print("✅ Kaggle working directories ready")

## Cell 2: Install vLLM & Launch Server (Conservative Settings)

In [ ]:
# Cell 2: Install vLLM & Launch Server (Conservative Settings)
!pip install -q vllm==0.6.4

import subprocess, time, urllib.request, pathlib, os

# Kill any existing vllm
subprocess.run(["pkill", "-f", "api_server"], capture_output=True)
time.sleep(2)

# Kaggle: use /kaggle/working for persistent logs
LOG = pathlib.Path("/kaggle/working/vllm_logs/vllm.log")
LOG.parent.mkdir(parents=True, exist_ok=True)
LOG.write_text("")

# ═══════════════════════════════════════════════════════════
# OPTIMIZED FOR KAGGLE 2× T4 (32GB TOTAL VRAM) — TENSOR PARALLEL
# ═══════════════════════════════════════════════════════════
vllm_proc = subprocess.Popen(
    [
        "python", "-m", "vllm.entrypoints.openai.api_server",
        "--model", "Qwen/Qwen2-VL-7B-Instruct-AWQ",
        "--quantization", "awq",
        "--host", "0.0.0.0", "--port", "8000",
        "--dtype", "half",
        "--tensor-parallel-size", "2",          # ← USE BOTH T4s (32GB total)
        "--gpu-memory-utilization", "0.90",      # ← 0.9 × 16GB × 2 = 28.8GB usable
        "--max-model-len", "16384",              # ← Double context (16K for VLM tasks)
        "--max-num-seqs", "16",                 # ← 4× concurrency
        "--max-num-batched-tokens", "16384",    # ← Match context window
        "--enforce-eager",
        "--trust-remote-code",
        "--limit-mm-per-prompt", "image=4",  # ← More images per prompt
        "--disable-frontend-multiprocessing",
        "--download-dir", "/kaggle/working/models"  # ← persist model cache
    ],
    stdout=open(LOG, "w"),
    stderr=subprocess.STDOUT,
    start_new_session=True,   # ← detaches from cell lifecycle
)

print(f"vLLM PID {vllm_proc.pid} — polling health...\n")

for i in range(90):
    time.sleep(10)
    lines = LOG.read_text().splitlines()
    if i % 3 == 0:
        print(f"t+{(i+1)*10}s:", lines[-1] if lines else "(silent)")
    if not pathlib.Path(f"/proc/{vllm_proc.pid}").exists():
        print("💀 Process died — log:")
        print("\n".join(lines[-30:]))
        break
    try:
        if urllib.request.urlopen("http://localhost:8000/health", timeout=2).status == 200:
            print(f"\n✅ vLLM healthy — PID {vllm_proc.pid} will survive cell exit")
            break
    except:
        pass
else:
    print(LOG.read_text()[-3000:])


## Cell 3: Rate-Limiting Proxy (Protects Against Public Abuse)

In [ ]:
import subprocess, os, time

proxy_code = '''
import time, threading
from collections import defaultdict
from fastapi import FastAPI, Request, HTTPException
from fastapi.responses import StreamingResponse
import httpx, uvicorn

app = FastAPI()
VLLM_URL = "http://localhost:8000"

# Simple per-IP rate limiter: 60 req/min (2× T4 with tensor parallel — 4× concurrency)
request_log = defaultdict(list)
lock = threading.Lock()
RPM_LIMIT = 60

def check_rate(ip):
    now = time.time()
    with lock:
        request_log[ip] = [t for t in request_log[ip] if now - t < 60]
        if len(request_log[ip]) >= RPM_LIMIT:
            return False
        request_log[ip].append(now)
        return True

@app.api_route("/" + "{path:path}", methods=["GET","POST","PUT","DELETE"])
async def proxy(path: str, request: Request):
    ip = request.client.host
    if not check_rate(ip):
        raise HTTPException(429, "Rate limit: 60 req/min")
    body = await request.body()
    async with httpx.AsyncClient(timeout=300) as client:
        resp = await client.request(
            method=request.method,
            url=f"{VLLM_URL}/{path}",
            headers=dict(request.headers),
            content=body,
        )
    return StreamingResponse(
        iter([resp.content]),
        status_code=resp.status_code,
        headers=dict(resp.headers)
    )

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8001)
'''


with open("/kaggle/working/vllm_logs/proxy.py", "w") as f:
    f.write(proxy_code)

subprocess.run(["pip", "install", "-q", "fastapi", "uvicorn", "httpx"], check=True)

subprocess.run(["pkill", "-f", "proxy.py"], capture_output=True)
time.sleep(1)

proxy_proc = subprocess.Popen(
    ["python", "/kaggle/working/vllm_logs/proxy.py"],
    stdout=open("/kaggle/working/vllm_logs/proxy.log", "w"),
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
print(f"✅ Rate-limiting proxy PID {proxy_proc.pid} on :8001")

## Cell 4: Cloudflare Tunnel (Points to Proxy :8001)

In [ ]:
import subprocess, re, time, os

if not os.path.exists("./cloudflared"):
    subprocess.run([
        "wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", "./cloudflared"
    ], check=True)
    os.chmod("./cloudflared", 0o755)

subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)
time.sleep(1)

# Point tunnel to PROXY port 8001, NOT vLLM directly
cf = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8001"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    start_new_session=True,
)

url = None
for _ in range(30):
    line = cf.stdout.readline()
    if line: print(line.strip())
    m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0)
        print(f"\n✅ URL:  {url}")
        print(f"   v1:   {url}/v1")
        print(f"\n   Verify: curl {url}/v1/models")
        break
    time.sleep(1)

# Store URL for later cells
with open("/kaggle/working/vllm_logs/tunnel_url.txt", "w") as f:
    f.write(url or "")
print(f"✅ Tunnel URL saved to /kaggle/working/vllm_logs/tunnel_url.txt")

## Cell 5: Verify End-to-End (with DNS retry logic)

In [ ]:
import urllib.request, json, re, subprocess, time, socket, ctypes, os

# Re-extract URL from saved file or running cloudflared
print("🔍 Finding active Cloudflare tunnel...")

# First try saved URL
url = None
if os.path.exists("/kaggle/working/vllm_logs/tunnel_url.txt"):
    with open("/kaggle/working/vllm_logs/tunnel_url.txt") as f:
        url = f.read().strip()

if not url:
    # Restart tunnel and extract
    subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)
    time.sleep(2)

    if not os.path.exists("./cloudflared"):
        subprocess.run([
            "wget", "-q",
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            "-O", "./cloudflared"
        ], check=True)
        os.chmod("./cloudflared", 0o755)

    cf = subprocess.Popen(
        ["./cloudflared", "tunnel", "--url", "http://localhost:8001"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        start_new_session=True,
    )

    for _ in range(60):
        line = cf.stdout.readline()
        if line:
            print(line.strip())
        m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if m:
            url = m.group(0)
            print(f"\n✅ URL captured: {url}")
            break
        time.sleep(1)

if not url:
    raise RuntimeError("❌ Could not extract Cloudflare URL")

# Save for persistence
with open("/kaggle/working/vllm_logs/tunnel_url.txt", "w") as f:
    f.write(url)

# ═══════════════════════════════════════════════════
# FIX: Wait for DNS propagation + force resolver refresh
# ═══════════════════════════════════════════════════
print(f"\n⏳ Waiting for DNS propagation (trycloudflare needs 30-90s)...")

# Force glibc to re-read /etc/resolv.conf (clears negative DNS cache)
try:
    libc = ctypes.cdll.LoadLibrary('libc.so.6')
    libc.__res_init()
    print("🔄 DNS resolver cache flushed")
except Exception as e:
    print(f"⚠️ Could not flush DNS cache: {e}")

# Retry with exponential backoff
max_retries = 15
base_delay = 3
for attempt in range(1, max_retries + 1):
    # Re-flush DNS before each attempt
    try:
        libc.__res_init()
    except:
        pass

    try:
        print(f"\n🌐 Attempt {attempt}/{max_retries}: Testing {url}/v1/models")
        req = urllib.request.Request(
            f"{url}/v1/models",
            headers={"Content-Type": "application/json"}
        )
        resp = json.loads(urllib.request.urlopen(req, timeout=15).read())
        print("\n✅ SUCCESS!")
        print(json.dumps(resp, indent=2))
        break
    except urllib.error.URLError as e:
        print(f"   ❌ Failed: {e}")
        if attempt == max_retries:
            print("\n💀 All retries exhausted. Possible causes:")
            print("   - Cloudflare tunnel died (check /kaggle/working/vllm_logs/vllm.log)")
            print("   - Proxy on :8001 is not running")
            print("   - trycloudflare.com is temporarily down")
            print("   - Kaggle network restrictions")
            raise
        delay = min(base_delay * (2 ** (attempt - 1)), 60)  # cap at 60s
        print(f"   ⏱️  Retrying in {delay}s...")
        time.sleep(delay)

## Cell 6: Keep Alive + Idle GPU Auto-Pause + Session Monitor (RUN THIS LAST)

In [ ]:
import urllib.request, subprocess, pathlib, time, os, signal

idle_since = None
IDLE_SHUTDOWN_MINUTES = 15  # Shorter for Kaggle quota
SESSION_MAX_HOURS = 8.5     # Leave buffer before 9hr hard limit
session_start = time.time()

print(f"🕐 Session monitor started — max {SESSION_MAX_HOURS}hr, idle shutdown {IDLE_SHUTDOWN_MINUTES}min")

while True:
    time.sleep(60)
    now = time.strftime("%H:%M:%S")
    elapsed_hours = (time.time() - session_start) / 3600

    # Session time limit check
    if elapsed_hours > SESSION_MAX_HOURS:
        print(f"🕐 Session limit reached ({elapsed_hours:.1f}hr) — shutting down gracefully")
        subprocess.run(["pkill", "-f", "api_server"], capture_output=True)
        subprocess.run(["pkill", "-f", "proxy.py"], capture_output=True)
        subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)
        break

    # Health check
    try:
        r = urllib.request.urlopen("http://localhost:8000/health", timeout=5)
        print(f"♥ Heartbeat {now} — Health: ✅ {r.status} | Session: {elapsed_hours:.1f}hr")
    except Exception as e:
        print(f"♥ Heartbeat {now} — Health: ❌ {e}")

    # Idle detection via metrics
    try:
        metrics = urllib.request.urlopen("http://localhost:8000/metrics", timeout=5).read().decode()
        running_lines = [l for l in metrics.splitlines() if "vllm:num_requests_running" in l and not l.startswith("#")]
        if running_lines:
            running = int(running_lines[-1].split()[-1])
            if running == 0:
                if idle_since is None:
                    idle_since = time.time()
                    print(f"   💤 GPU idle detected...")
                elif (time.time() - idle_since) > IDLE_SHUTDOWN_MINUTES * 60:
                    print(f"   💤 Idle {IDLE_SHUTDOWN_MINUTES}min — pausing vLLM to save GPU hours")
                    subprocess.run(["pkill", "-STOP", "-f", "api_server"], capture_output=True)
                    print(f"   ⏸️  vLLM paused. Resume on next request.")
            else:
                if idle_since:
                    subprocess.run(["pkill", "-CONT", "-f", "api_server"], capture_output=True)
                    print(f"   ▶️  vLLM resumed — requests active")
                idle_since = None
    except Exception as e:
        pass

    # Show last log lines
    log = pathlib.Path("/kaggle/working/vllm_logs/vllm.log")
    if log.exists():
        lines = log.read_text().splitlines()
        print(f"\n--- Last 5 log lines ({len(lines)} total) ---")
        print("\n".join(lines[-5:]))

    print("═" * 50)